In [ ]:
# CELL 1: SETUP & WIDGETS
# -------------------------------------------------------------------------
# Configure the catalog, schema, and scenario ID to analyze
# -------------------------------------------------------------------------

dbutils.widgets.text("CATALOG", "sample_synthetic_sap", "Catalog")
dbutils.widgets.text("SCHEMA", "sap", "Schema")
dbutils.widgets.text("SCENARIO_ID", "", "Scenario ID (e.g., SCN001)")

CATALOG = dbutils.widgets.get("CATALOG")
SCHEMA = dbutils.widgets.get("SCHEMA")
SCENARIO_ID = dbutils.widgets.get("SCENARIO_ID").upper().strip()

print(f"Connected to: {CATALOG}.{SCHEMA}")
if not SCENARIO_ID:
    print("Please enter a SCENARIO_ID in the widget above (e.g., SCN001)")
else:
    print(f"Analyzing Scenario: {SCENARIO_ID}")

In [ ]:
# CELL 2: LIBRARIES
# -------------------------------------------------------------------------
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from datetime import datetime

In [ ]:
# CELL 3: SCENARIO DESCRIPTOR TABLE
# -------------------------------------------------------------------------
# Load scenario configuration: What, When, Where
# -------------------------------------------------------------------------

print("=" * 70)
print("SCENARIO CONFIGURATION")
print("=" * 70)

if not SCENARIO_ID:
    print("Please enter a SCENARIO_ID in the widget above.")
else:
    try:
        df_config = spark.table(f"{CATALOG}.{SCHEMA}.scenario_config")
        df_selected = df_config.filter(F.col("SCENARIO_ID") == SCENARIO_ID)
        
        if df_selected.count() == 0:
            print(f"Scenario {SCENARIO_ID} not found. Available scenarios:")
            display(df_config.select("SCENARIO_ID", "SCENARIO_NAME", "SCENARIO_TYPE").orderBy("SCENARIO_ID"))
        else:
            df_summary = df_selected.select(
                F.col("SCENARIO_ID"),
                F.col("SCENARIO_NAME"),
                F.col("SCENARIO_TYPE"),
                F.col("DESCRIPTION"),
                F.col("IMPACT_DATE").alias("When"),
                F.col("IMPACT_DURATION_DAYS").alias("Duration_Days"),
                F.col("IMPACTED_NODE").alias("Where"),
                F.col("IMPACTED_PRODUCTS").alias("What_Products"),
                F.col("INVENTORY_IMPACT"),
                F.col("INVENTORY_QTY")
            )
            display(df_summary)
    except Exception as e:
        print(f"Error loading scenario_config table: {str(e)}")
        print("Make sure the Setup Scenario Config notebook has been run.")

In [ ]:
# CELL 4: SCENARIO DETAILS
# -------------------------------------------------------------------------
# Detailed breakdown: When, Where, What, AI Options, Evidence
# -------------------------------------------------------------------------

scenario_row = []
if SCENARIO_ID:
    try:
        scenario_row = df_selected.collect()
        
        if scenario_row:
            s = scenario_row[0]
            
            print("=" * 70)
            print(f"SCENARIO: {s['SCENARIO_ID']} - {s['SCENARIO_NAME']}")
            print("=" * 70)
            
            print(f"\nDESCRIPTION:")
            print(f"   {s['DESCRIPTION']}")
            
            print(f"\nWHEN (TIME):")
            impact_date = s['IMPACT_DATE']
            if impact_date:
                formatted_date = f"{impact_date[0:4]}-{impact_date[4:6]}-{impact_date[6:8]}"
                print(f"   Impact Date: {formatted_date}")
            duration = s['IMPACT_DURATION_DAYS']
            if duration and duration > 0:
                print(f"   Duration: {duration} days")
            else:
                print(f"   Duration: {'Permanent' if s['IMPACT_PERMANENT'] else 'Instantaneous'}")
            
            print(f"\nWHERE (LOCATION):")
            print(f"   Node/Plant: {s['IMPACTED_NODE']}")
            if s['NODE_OFFLINE']:
                print(f"   Node Status: OFFLINE")
            else:
                print(f"   Node Status: Online (Capacity: {s['NODE_CAPACITY_PCT']}%)")
            if s['TLANES_AFFECTED']:
                print(f"   Transportation Lanes: AFFECTED")
            
            print(f"\nWHAT (PRODUCTS):")
            print(f"   Materials: {s['IMPACTED_PRODUCTS']}")
            if s['IMPACTED_BATCH'] and s['IMPACTED_BATCH'] != 'N/A':
                print(f"   Batches: {s['IMPACTED_BATCH']}")
            if s['INVENTORY_IMPACT'] and s['INVENTORY_IMPACT'] != 'NONE':
                print(f"   Inventory Impact: {s['INVENTORY_IMPACT']} - Qty: {s['INVENTORY_QTY']}")
            
            print(f"\nAI DECISION OPTIONS:")
            options = s['AI_DECISION_OPTIONS']
            if options:
                for opt in options.split('|'):
                    print(f"   - {opt.strip()}")
            
            print(f"\nDATA EVIDENCE:")
            evidence = s['DATA_EVIDENCE']
            if evidence:
                for ev in evidence.split('|'):
                    print(f"   > {ev.strip()}")
            
            print("\n" + "=" * 70)
    except Exception as e:
        print(f"Error: {str(e)}")

In [ ]:
# CELL 5: INVENTORY IMPACT VISUALIZATION
# -------------------------------------------------------------------------
# Daily inventory - reconstructs from MARD (current) working backwards
# -------------------------------------------------------------------------

print("=" * 70)
print("INVENTORY TREND (Daily)")
print("=" * 70)

if SCENARIO_ID and scenario_row:
    s = scenario_row[0]
    impacted_node = s['IMPACTED_NODE']
    impacted_products = s['IMPACTED_PRODUCTS']
    impact_date = s['IMPACT_DATE']
    duration_days = s['IMPACT_DURATION_DAYS'] or 0

    # Parse materials list
    if impacted_products and impacted_products != 'ALL':
        materials = [m.strip() for m in impacted_products.split(',')]
    else:
        materials = None

    # Calculate display date range
    import pandas as pd
    if impact_date:
        impact_date_obj = pd.Timestamp(f"{impact_date[0:4]}-{impact_date[4:6]}-{impact_date[6:8]}")
        display_start = impact_date_obj - pd.Timedelta(days=30)
        display_end = impact_date_obj + pd.Timedelta(days=max(duration_days, 1) + 30)
        print(f"Display range: {display_start.strftime('%Y-%m-%d')} to {display_end.strftime('%Y-%m-%d')}")
    else:
        display_start = None
        display_end = None

    # Get CURRENT stock from MARD (this is our known reference point)
    df_mard = spark.table(f"{CATALOG}.{SCHEMA}.mard")
    df_mard_filtered = df_mard

    if impacted_node and impacted_node != 'ALL':
        df_mard_filtered = df_mard_filtered.filter(F.col("WERKS") == impacted_node)

    if materials:
        df_mard_filtered = df_mard_filtered.filter(F.col("MATNR").isin(materials))

    current_stock = df_mard_filtered.agg(
        F.sum("LABST").alias("current_stock")
    ).collect()[0]["current_stock"] or 0
    
    print(f"Current stock (MARD): {current_stock:,.0f}")

    # Get ALL movements from MATDOC for this material/plant
    df_matdoc = spark.table(f"{CATALOG}.{SCHEMA}.matdoc")
    df_matdoc_filtered = df_matdoc

    if impacted_node and impacted_node != 'ALL':
        df_matdoc_filtered = df_matdoc_filtered.filter(F.col("WERKS") == impacted_node)

    if materials:
        df_matdoc_filtered = df_matdoc_filtered.filter(F.col("MATNR").isin(materials))

    # Daily aggregation of ALL movements
    df_daily_all = (
        df_matdoc_filtered
        .withColumn("Date", F.to_date(F.col("BUDAT"), "yyyyMMdd"))
        .withColumn("Flow_Qty",
            F.when(F.col("SHKZG") == "S", F.col("MENGE")).otherwise(F.col("MENGE") * -1)
        )
        .groupBy("Date")
        .agg(
            F.sum(F.when(F.col("Flow_Qty") > 0, F.col("Flow_Qty")).otherwise(0)).alias("Daily_Incoming"),
            F.sum(F.when(F.col("Flow_Qty") < 0, F.col("Flow_Qty")).otherwise(0)).alias("Daily_Outgoing"),
            F.sum("Flow_Qty").alias("Daily_Net")
        )
        .orderBy("Date")
    )

    # Calculate total net movements
    total_net = df_daily_all.agg(F.sum("Daily_Net")).collect()[0][0] or 0
    
    # Derive starting stock: current = starting + total_net, so starting = current - total_net
    starting_stock = float(current_stock) - float(total_net)
    print(f"Total net movements: {total_net:,.0f}")
    print(f"Implied starting stock: {starting_stock:,.0f}")

    # Calculate cumulative stock forward from starting
    window_spec = Window.orderBy("Date").rowsBetween(Window.unboundedPreceding, Window.currentRow)
    
    df_inventory_all = df_daily_all.withColumn(
        "Stock_Balance",
        F.lit(starting_stock) + F.sum("Daily_Net").over(window_spec)
    )

    # Now filter to display range
    if display_start and display_end:
        df_inventory_display = df_inventory_all.filter(
            (F.col("Date") >= display_start) & (F.col("Date") <= display_end)
        )
    else:
        df_inventory_display = df_inventory_all

    pdf = df_inventory_display.toPandas()

    if not pdf.empty:
        print(f"Stock range in window: {pdf['Stock_Balance'].min():,.0f} to {pdf['Stock_Balance'].max():,.0f}")
        
        impact_date_ts = impact_date_obj if impact_date else None

        fig = go.Figure()

        # Demand (Red bars)
        fig.add_trace(go.Bar(
            x=pdf['Date'], y=pdf['Daily_Outgoing'],
            name='Demand/Issues', marker_color='salmon', opacity=0.7
        ))

        # Supply (Green bars)
        fig.add_trace(go.Bar(
            x=pdf['Date'], y=pdf['Daily_Incoming'],
            name='Supply/Receipts', marker_color='mediumseagreen', opacity=0.7
        ))

        # Stock Line (Blue)
        fig.add_trace(go.Scatter(
            x=pdf['Date'], y=pdf['Stock_Balance'],
            mode='lines', name='Stock Balance',
            line=dict(color='royalblue', width=2)
        ))

        # Scenario impact date marker
        if impact_date_ts is not None:
            fig.add_shape(
                type="line",
                x0=impact_date_ts, x1=impact_date_ts,
                y0=0, y1=1,
                yref="paper",
                line=dict(color="red", width=3, dash="dash")
            )
            fig.add_annotation(
                x=impact_date_ts, y=1.05,
                yref="paper",
                text=f"{SCENARIO_ID} Impact",
                showarrow=False,
                font=dict(color="red", size=12)
            )

        location_text = impacted_node if impacted_node != 'ALL' else 'All Plants'
        material_text = impacted_products if impacted_products != 'ALL' else 'All Materials'

        fig.update_layout(
            title=f"Daily Inventory: {SCENARIO_ID} - {location_text} / {material_text}",
            xaxis_title="Date",
            yaxis_title="Quantity",
            template="plotly_white",
            hovermode="x unified",
            barmode='overlay'
        )

        fig.show()
    else:
        print(f"No inventory data found for the scenario filters and date range.")
else:
    print("Enter a SCENARIO_ID to see inventory visualization.")

In [ ]:
# CELL 6: SUPPLY CHAIN VALUE TREND
# -------------------------------------------------------------------------
# Weekly supply chain value with scenario impact highlighted
# -------------------------------------------------------------------------

print("=" * 70)
print("SUPPLY CHAIN VALUE TREND")
print("=" * 70)

if SCENARIO_ID and scenario_row:
    s = scenario_row[0]
    impacted_node = s['IMPACTED_NODE']
    impact_date = s['IMPACT_DATE']
    
    # Parse impact date for vertical line
    import pandas as pd
    impact_date_ts = None
    if impact_date:
        try:
            impact_date_ts = pd.Timestamp(f"{impact_date[0:4]}-{impact_date[4:6]}-{impact_date[6:8]}")
        except:
            pass
    
    # Load data
    df_matdoc = spark.table(f"{CATALOG}.{SCHEMA}.matdoc")
    df_mbew = spark.table(f"{CATALOG}.{SCHEMA}.mbew")
    df_mara = spark.table(f"{CATALOG}.{SCHEMA}.mara")
    
    # Price lookup
    df_prices = (
        df_mbew.alias("v")
        .join(df_mara.alias("m"), F.col("v.MATNR") == F.col("m.MATNR"), how="left")
        .select(
            F.col("v.MATNR"),
            F.col("v.BWKEY").alias("WERKS"),
            F.coalesce(F.col("v.STPRS"), F.lit(0)).alias("Price"),
            F.col("m.MTART").alias("Material_Type")
        )
    )
    
    # Filter by location if specified
    df_filtered = df_matdoc
    if impacted_node and impacted_node != 'ALL':
        df_filtered = df_filtered.filter(F.col("WERKS") == impacted_node)
    
    # Calculate weekly stock movements
    df_weekly_moves = (
        df_filtered
        .withColumn("Week_End", F.date_trunc("week", F.to_date(F.col("BUDAT"), "yyyyMMdd")) + F.expr("INTERVAL 6 DAYS"))
        .withColumn("Qty_Signed",
                    F.when(F.col("SHKZG") == "S", F.col("MENGE"))
                     .otherwise(F.col("MENGE") * -1))
        .groupBy("MATNR", "WERKS", "Week_End")
        .agg(F.sum("Qty_Signed").alias("Weekly_Movement"))
    )
    
    # Running stock balance
    window_spec = Window.partitionBy("MATNR", "WERKS").orderBy("Week_End").rowsBetween(Window.unboundedPreceding, Window.currentRow)
    
    df_weekly_stock = (
        df_weekly_moves
        .withColumn("Stock_Balance", F.sum("Weekly_Movement").over(window_spec))
        .filter(F.col("Stock_Balance") > 0)
        .join(df_prices, on=["MATNR", "WERKS"], how="left")
        .withColumn("Stock_Value", F.col("Stock_Balance") * F.coalesce(F.col("Price"), F.lit(0)))
    )
    
    # Weekly total
    df_weekly_total = (
        df_weekly_stock
        .groupBy("Week_End")
        .agg(F.sum("Stock_Value").alias("Total_Value_EUR"))
        .orderBy("Week_End")
    )
    
    pdf_total = df_weekly_total.toPandas()
    
    if not pdf_total.empty:
        fig = go.Figure()
        
        fig.add_trace(go.Scatter(
            x=pdf_total['Week_End'],
            y=pdf_total['Total_Value_EUR'],
            mode='lines+markers',
            name='Total Value',
            line=dict(color='#2E86AB', width=3),
            marker=dict(size=6),
            fill='tozeroy',
            fillcolor='rgba(46, 134, 171, 0.2)'
        ))
        
        # Scenario impact date marker using add_shape
        if impact_date_ts is not None:
            fig.add_shape(
                type="line",
                x0=impact_date_ts, x1=impact_date_ts,
                y0=0, y1=1,
                yref="paper",
                line=dict(color="red", width=3, dash="dash")
            )
            fig.add_annotation(
                x=impact_date_ts, y=1.05,
                yref="paper",
                text=f"{SCENARIO_ID} Impact",
                showarrow=False,
                font=dict(color="red", size=12)
            )
        
        location_text = impacted_node if impacted_node != 'ALL' else 'All Plants'
        
        fig.update_layout(
            title=f'Weekly Supply Chain Value (EUR) - {location_text}',
            xaxis_title='Week Ending',
            yaxis_title='Total Value (EUR)',
            template='plotly_white',
            hovermode='x unified',
            yaxis_tickformat=',.0f',
            yaxis_tickprefix='\u20ac'
        )
        
        fig.show()
    else:
        print("No data available for visualization.")
else:
    print("Enter a SCENARIO_ID to see value trend visualization.")

In [ ]:
# CELL 7: VALUE BY MATERIAL TYPE (STACKED AREA)
# -------------------------------------------------------------------------
# Inventory value distribution across material types over time
# -------------------------------------------------------------------------

print("=" * 70)
print("VALUE BY MATERIAL TYPE")
print("=" * 70)

if SCENARIO_ID and scenario_row:
    df_weekly_by_type = (
        df_weekly_stock
        .groupBy("Week_End", "Material_Type")
        .agg(F.sum("Stock_Value").alias("Total_Value_EUR"))
        .orderBy("Week_End", "Material_Type")
    )
    
    pdf_type_trend = df_weekly_by_type.toPandas()
    
    if not pdf_type_trend.empty:
        pdf_pivot = pdf_type_trend.pivot(index='Week_End', columns='Material_Type', values='Total_Value_EUR').fillna(0)
        
        fig = go.Figure()
        
        colors = {'FERT': '#2E86AB', 'HALB': '#A23B72', 'ROH': '#F18F01'}
        type_names = {'FERT': 'Finished Goods', 'HALB': 'Semi-Finished', 'ROH': 'Raw Materials'}
        
        for mat_type in pdf_pivot.columns:
            fig.add_trace(go.Scatter(
                x=pdf_pivot.index,
                y=pdf_pivot[mat_type],
                mode='lines',
                name=type_names.get(mat_type, mat_type),
                stackgroup='one',
                line=dict(width=0.5),
                fillcolor=colors.get(mat_type, '#888888')
            ))
        
        # Scenario impact date marker using add_shape
        if impact_date_ts is not None:
            fig.add_shape(
                type="line",
                x0=impact_date_ts, x1=impact_date_ts,
                y0=0, y1=1,
                yref="paper",
                line=dict(color="red", width=3, dash="dash")
            )
            fig.add_annotation(
                x=impact_date_ts, y=1.05,
                yref="paper",
                text=f"{SCENARIO_ID}",
                showarrow=False,
                font=dict(color="red", size=12)
            )
        
        location_text = impacted_node if impacted_node != 'ALL' else 'All Plants'
        
        fig.update_layout(
            title=f'Weekly Value by Material Type (EUR) - {location_text}',
            xaxis_title='Week Ending',
            yaxis_title='Total Value (EUR)',
            template='plotly_white',
            hovermode='x unified',
            yaxis_tickformat=',.0f',
            yaxis_tickprefix='\u20ac',
            legend_title='Material Type'
        )
        
        fig.show()
    else:
        print("No data available for material type visualization.")
else:
    print("Enter a SCENARIO_ID to see material type breakdown.")

In [ ]:
# APPENDIX: SCENARIO MATDOC RECORDS
# -------------------------------------------------------------------------
# Raw material document records injected for this scenario
# -------------------------------------------------------------------------

print("=" * 70)
print("APPENDIX: SCENARIO MATERIAL DOCUMENTS")
print("=" * 70)

if SCENARIO_ID:
    df_matdoc = spark.table(f"{CATALOG}.{SCHEMA}.matdoc")
    df_scenario_docs = df_matdoc.filter(F.col("MBLNR").startswith(SCENARIO_ID))
    
    df_scenario_display = df_scenario_docs.select(
        F.col("MBLNR").alias("Document_Number"),
        F.col("BWART").alias("Movement_Type"),
        F.col("MATNR").alias("Material"),
        F.col("WERKS").alias("Plant"),
        F.col("LGORT").alias("Storage_Loc"),
        F.col("MENGE").alias("Quantity"),
        F.col("SHKZG").alias("Debit_Credit"),
        F.col("BUDAT").alias("Posting_Date"),
        F.col("BKTXT").alias("Description")
    ).orderBy("Document_Number")
    
    count = df_scenario_display.count()
    print(f"\nFound {count} material documents for {SCENARIO_ID}.")
    
    if count > 0:
        display(df_scenario_display)
        
        # Movement type summary
        print("\n--- Movement Type Summary ---")
        df_mvmt_summary = df_scenario_docs.groupBy("BWART").agg(
            F.count("*").alias("Count"),
            F.sum("MENGE").alias("Total_Qty")
        ).orderBy("BWART")
        display(df_mvmt_summary)
    else:
        print("No scenario records found. Run the Inject Scenarios notebook first.")
else:
    print("Enter a SCENARIO_ID to see material documents.")